# InternVL3 MELD Benchmark

Kaggle notebook pipeline for extracting frozen InternVL3 shared video+utterance embeddings, training the MLP probe, and inspecting labeled metrics.

In [ ]:
# install packages first
!pip install -r /kaggle/input/datasets/pushkarsingh2005/benchmark-code/requirements.txt

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

In [ ]:
# paths
CODE = "/kaggle/input/datasets/pushkarsingh2005/benchmark-code"

INDEX_DIR = f"{CODE}/outputs/indexes"

EMB_ROOT = "/kaggle/working/embeddings/internvl3_2b_shared"
CHUNK_ROOT = "/kaggle/working/embeddings/internvl3_2b_shared_chunks"
OUT_DIR = "/kaggle/working/outputs/internvl3_2b_shared"

In [ ]:
# sanity check indexes
import pandas as pd

for split in ["train", "dev", "test"]:
    path = f"{INDEX_DIR}/meld_{split}_index.csv"
    df = pd.read_csv(path)
    print(split, df.shape)
    print(df[["sample_id", "video_path", "emotion"]].head(2))
    print()

In [ ]:
# extract InternVL3 chunks
import subprocess

for split in ["train", "dev", "test"]:
    cmd = [
        "python", f"{CODE}/scripts/run_internvl3_chunks.py",
        "--index-csv", f"{INDEX_DIR}/meld_{split}_index.csv",
        "--chunks-dir", f"{CHUNK_ROOT}/{split}",
        "--chunk-prefix", split,
        "--chunk-size", "500",
        "--fps", "6",
        "--max-frames", "64",
        "--pooling", "last",
        "--prompt-style", "emotion_task",
        "--modality-mode", "video_text",
        "--save-dtype", "float32",
        "--gc-every", "5",
        "--skip-existing-complete",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
# merge chunks
import subprocess

for split in ["train", "dev", "test"]:
    cmd = [
        "python", f"{CODE}/scripts/merge_embedding_chunks.py",
        "--chunks-dir", f"{CHUNK_ROOT}/{split}",
        "--output-pt", f"{EMB_ROOT}/meld_{split}.pt",
        "--pattern", f"{split}_*.pt",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
# check merged embeddings
import torch

for split in ["train", "dev", "test"]:
    payload = torch.load(f"{EMB_ROOT}/meld_{split}.pt", map_location="cpu", weights_only=False)
    print(split)
    print("embeddings:", tuple(payload["embeddings"].shape))
    print("labels:", tuple(payload["labels"].shape))
    print("errors:", len(payload.get("errors", [])))
    print("config embedding_type:", payload.get("config", {}).get("embedding_type"))
    print()

In [ ]:
# train MLP probe
import subprocess

cmd = [
    "python", f"{CODE}/scripts/train_mlp.py",
    "--train-pt", f"{EMB_ROOT}/meld_train.pt",
    "--dev-pt", f"{EMB_ROOT}/meld_dev.pt",
    "--test-pt", f"{EMB_ROOT}/meld_test.pt",
    "--output-dir", OUT_DIR,
    "--hidden-dim", "512",
    "--epochs", "50",
    "--device", "cuda",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# main metrics
import json

with open(f"{OUT_DIR}/metrics.json", "r") as f:
    metrics = json.load(f)

print("Best dev:")
print("Accuracy:", metrics["best_dev"]["accuracy"])
print("Macro F1:", metrics["best_dev"]["macro_f1"])
print("Weighted F1:", metrics["best_dev"]["weighted_f1"])

print("\nTest:")
print("Accuracy:", metrics["test"]["accuracy"])
print("Macro F1:", metrics["test"]["macro_f1"])
print("Weighted F1:", metrics["test"]["weighted_f1"])

In [ ]:
# confusion matrices
import pandas as pd

DEFAULT_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
labels = metrics.get("label_names", DEFAULT_LABELS)

def labeled_confusion_matrix(split_metrics):
    return pd.DataFrame(
        split_metrics["confusion_matrix"],
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    )

dev_cm = labeled_confusion_matrix(metrics["best_dev"])
test_cm = labeled_confusion_matrix(metrics["test"])

print("Label order:", labels)
print("Rows are true labels; columns are predicted labels.")

display(dev_cm)
display(test_cm)


In [ ]:
# classification report tables
dev_report = pd.DataFrame(metrics["best_dev"]["classification_report"]).T
test_report = pd.DataFrame(metrics["test"]["classification_report"]).T

display(dev_report)
display(test_report)